# 第13章 RAG

## 13.3 RAG 向けに LLM を指示チューニングする

### 13.3.1 AI 王データセットを用いた指示チューニング

#### 環境の準備

In [1]:
!pip install 'datasets<4.0.0' transformers[torch,sentencepiece] trl peft bitsandbytes

INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.7 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from transformers.trainer_utils import set_seed
set_seed(42)

In [3]:
from google.colab import drive
drive.mount("drive")

Mounted at drive


#### データセットの準備

In [4]:
from datasets import load_dataset

# AI王データセットの読み込み
dataset = load_dataset(
    "llm-book/aio-retriever", trust_remote_code=True
)

README.md:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

aio-retriever.py:   0%|          | 0.00/3.58k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22335 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['qid', 'competition', 'timestamp', 'section', 'number', 'original_question', 'original_answer', 'original_additional_info', 'question', 'answers', 'passages', 'positive_passage_indices', 'negative_passage_indices'],
        num_rows: 22335
    })
    validation: Dataset({
        features: ['qid', 'competition', 'timestamp', 'section', 'number', 'original_question', 'original_answer', 'original_additional_info', 'question', 'answers', 'passages', 'positive_passage_indices', 'negative_passage_indices'],
        num_rows: 1000
    })
})


In [8]:
from pprint import pprint
pprint(dataset["validation"][0])

{'answers': ['ジェット団'],
 'competition': '第2回AI王',
 'negative_passage_indices': [1,
                              2,
                              4,
                              6,
                              8,
                              9,
                              10,
                              11,
                              13,
                              14,
                              15,
                              16,
                              17,
                              18,
                              19,
                              20,
                              21,
                              23,
                              24,
                              25,
                              26,
                              27,
                              28,
                              29,
                              30,
                              31,
                              32,
                              33,
     

In [9]:
from typing import Any

def filter_example(
    example: dict[str, Any],
    max_passages: int=3,
) -> bool:
    """上位max_passages件のパッセージに正例が含まれていない事例を除外"""
    if len(example["positive_passage_indices"]) == 0:
        return False
    if example["positive_passage_indices"][0] >= max_passages:
        return False
    return True

dataset = dataset.filter(filter_example)

Filter:   0%|          | 0/22335 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
pprint(dataset["validation"][1])

{'answers': ['コマイ'],
 'competition': '第2回AI王',
 'negative_passage_indices': [1,
                              4,
                              5,
                              6,
                              7,
                              8,
                              9,
                              10,
                              11,
                              12,
                              13,
                              14,
                              15,
                              16,
                              17,
                              18,
                              19,
                              20,
                              21,
                              22,
                              23,
                              24,
                              25,
                              26,
                              28,
                              29,
                              30,
                              31,
        

In [11]:
def process_example(
    example: dict[str, Any], max_passages: int=3
) -> dict[str, Any]:
    """質問、パッセージ、正解の組からプロンプトを作成し、会話データに変換"""

    # exampleから必要な情報を取得
    question = example["question"]
    answer = example["answers"][0]
    passages = [p["text"] for p in example["passages"]]

    # max_passages件のパッセージを選択
    passages = passages[:max_passages]

    messages: list[dict[str, str]] = []
    # promptとpassagesをユーザーのメッセージとして会話データに追加
    prompt_text = "".join(
        [
            "あなたには今からクイズに答えてもらいます。",
            "問題を与えますので、その解答のみを簡潔に出力してください。\n",
            "また解答の参考になりうるテキストを与えます。",
            "解答を含まない場合もあるのでその場合は無視してください。\n\n",
            "---\n",
            "\n\n".join(passages),
            "\n---\n\n",
            f"問題: {question}",

        ]
    )
    messages.append({"role": "user", "content": prompt_text})
    # LLMが出力すべき内容（クイズ問題の答え）を会話データに追加
    messages.append({"role": "assistant", "content": answer})

    # 会話データを事例の"messages"フィールドに追加
    example["messages"] = messages
    return example

dataset = dataset.map(
    process_example, remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/13951 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

In [12]:
# 前処理後のデータセットの形式と事例数を確認
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 13951
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 637
    })
})


In [13]:
# 前処理後のデータセットの内容を確認
pprint(dataset["validation"][0])

{'messages': [{'content': 'あなたには今からクイズに答えてもらいます。問題を与えますので、その解答のみを簡潔に出力してください。\n'
                          'また解答の参考になりうるテキストを与えます。解答を含まない場合もあるのでその場合は無視してください。\n'
                          '\n'
                          '---\n'
                          'ニューヨークのウエスト・サイド。午後5時。ポーランド系アメリカ人の少年非行グループ「ジェッツ」(ジェット団)と、新参のプエルトリコ系アメリカ人の少年非行グループ「シャークス」(シャーク団)は、なわばりを巡って対立している。今日も2グループの間で争いが起きるが警官の呼子笛の音に止められる(“Prologue”「プロローグ」)。クラプキ巡査とシュランク警部補が現れて少年たちに説教をして帰っていく。ジェッツのリーダー・リフはシャークスとの関係をはっきりさせるために決闘しようと言い出し、ジェッツのメンバーが賛成する。ついては決闘についての取り決めをシャークスとする必要があり、リフは自分の副官にトニーを選ぶ。メンバーは初めトニーはもう抜けたと反対するが、リフは(海兵隊のように)「一度ジェッツになったら死ぬまでジェッツだ」と歌う。\n'
                          '\n'
                          '『ウエストサイド物語』(ウエストサイドものがたり)は、宝塚歌劇団によるミュージカル作品。ブロードウェイ・ミュージカルの傑作『ウエストサイド物語』の日本での上演の一つである。\n'
                          '\n'
                          '『ウエスト・サイド物語』(ウエスト・サイドものがたり、West Side '
                          'Story)は、アーサー・ローレンツ脚本、レナード・バーンスタイン音楽、スティーヴン・ソンドハイム歌詞のブロードウェイ・ミュージカル。原案ジェローム・ロビンズ。1957年初演。『ウエスト・サイド・ストーリー』とも